# Pre-averaged Estimation

Pre-averaging smooths each asset's returns over a window of ticks before combining them, which cancels most of the noise, and pairs the smoothed returns of two assets wherever their windows overlap, as [Hayashi and Yoshida (2005)](https://doi.org/10.3150/bj/1116340299) do with raw returns. This tutorial runs the estimator of [Christensen, Podolskij and Vetter (2013)](https://doi.org/10.1016/j.jmva.2013.05.002) with `estimate_covariance(ts, assets, :preaveraged_covariance)` and tries several windows.

In [ ]:
Pollis.packages("HighFrequencyCovariance", "Plots")

## The Simulated Assets

Four assets with asynchronous ticks, noise and a known true covariance, as in the first tutorial.

In [ ]:
using HighFrequencyCovariance, LinearAlgebra, Random, Statistics
import Plots

ts, true_covar, true_noise, _ = generate_random_path(4, 10000; rng = MersenneTwister(2))  # four assets, 10,000 ticks in all, known true covariance
assets = get_assets(ts)

## The Pre-averaged Estimate

The window of k ticks is theta times the square root of the number of ticks, with `theta = 0.15` by default. The result is compared with realized covariance on the package's grid.

In [ ]:
pav = estimate_covariance(ts, assets, :preaveraged_covariance)
rc = estimate_covariance(ts, assets, :simple_covariance)
(pre_averaged = calculate_mean_abs_distance(pav, true_covar), realized = calculate_mean_abs_distance(rc, true_covar))

## The Choice of Theta

A small theta averages over few ticks and leaves noise in; a large theta smooths more but leaves fewer effective returns. The correlation error against theta shows the balance on this sample.

In [ ]:
thetas = [0.05, 0.1, 0.15, 0.3, 0.5, 1.0, 2.0]
errors = [calculate_mean_abs_distance(estimate_covariance(ts, assets, :preaveraged_covariance; theta = t), true_covar).Correlation_error for t in thetas]
Plots.plot(thetas, errors; xscale = :log10, marker = :circle, xlabel = "theta", ylabel = "correlation error", label = "pre-averaged", title = "Pre-averaged covariance against theta")

## Estimated Against True Correlations

Each off-diagonal correlation of the estimate against its true value: points on the diagonal line are estimated exactly.

In [ ]:
truth = Matrix(true_covar.correlation)
upper = [(i, j) for i in 1:4 for j in i+1:4]
Plots.scatter([truth[i, j] for (i, j) in upper], [Matrix(pav.correlation)[i, j] for (i, j) in upper]; xlabel = "true correlation", ylabel = "estimated correlation", label = "pre-averaged", legend = :topleft)
Plots.scatter!([truth[i, j] for (i, j) in upper], [Matrix(rc.correlation)[i, j] for (i, j) in upper]; label = "realized")
Plots.plot!([-1, 1], [-1, 1]; linestyle = :dash, label = "")

## Things to Try

- Draw a path with more ticks, 40,000, and see whether the best theta moves.
- Pass `regularisation = missing` to see the estimate before the package makes it positive semi-definite, as the fourth tutorial does.